# Early Stress Intervention Orchestrator## Notebook 1 of 3 — Synthetic Data GenerationGenerates 500 synthetic Australian mortgage customers with realisticincome, loan, and behavioural distributions based on ABS 2021 Censusdata and APRA serviceability thresholds.Outputs:- `/content/drive/MyDrive/mortgage-agent/data/customers.csv` (500 rows)- `/content/drive/MyDrive/mortgage-agent/data/weekly_signals.csv` (4,000 rows)- Both tables uploaded to Supabase**Run cells top to bottom.** Total runtime: ~3 minutes.

In [ ]:
# Cell 1 — Mount Google Drivefrom google.colab import drivedrive.mount('/content/drive')import osos.makedirs('/content/drive/MyDrive/mortgage-agent/data', exist_ok=True)print("Drive mounted. Files will persist between sessions.")

In [ ]:
# Cell 2 — Install dependencies!pip install faker supabase python-dotenv -qprint("Done")

In [ ]:
# Cell 3 — Generate 500 synthetic customersimport pandas as pdimport numpy as npfrom faker import Fakerfake = Faker('en_AU')np.random.seed(42)n = 500# ABS 2021 Census distributions for Australian mortgage holdersincomes = np.random.normal(95000, 28000, n).clip(45000, 250000)loan_sizes = np.random.normal(620000, 180000, n).clip(200000, 1500000)rates = np.random.uniform(0.0589, 0.0699, n)monthly_repayment = (loan_sizes * (rates/12)) / (1 - (1 + rates/12)**-300)def make_email(name):    parts = name.lower().split()    domains = ['gmail.com', 'outlook.com', 'hotmail.com',               'bigpond.com', 'icloud.com']    styles = [        f"{parts[0]}.{parts[-1]}",        f"{parts[0]}{parts[-1]}{np.random.randint(1, 99)}",        f"{parts[0][0]}{parts[-1]}",    ]    return f"{np.random.choice(styles)}@{np.random.choice(domains)}"names = [fake.name() for _ in range(n)]df = pd.DataFrame({    'customer_id': [f'CBA-{str(i).zfill(5)}' for i in range(n)],    'name': names,    'email': [make_email(name) for name in names],    'state': np.random.choice(['NSW', 'VIC', 'QLD', 'WA', 'SA'],        n, p=[0.32, 0.26, 0.20, 0.11, 0.11]),    'annual_income': incomes.round(0),    'loan_balance': loan_sizes.round(0),    'monthly_repayment': monthly_repayment.round(0),    'savings_balance': np.random.normal(18000, 12000, n).clip(0, 80000).round(0),    'credit_card_limit': np.random.choice([5000, 8000, 10000, 15000, 20000], n),    'credit_card_balance': np.random.normal(4200, 3000, n).clip(0, 20000).round(0),    'intervention_status': 'none',    'email_sent': False,    'email_opened': False,    'email_replied': False,})# APRA serviceability thresholds: 30% moderate, 45% severedf['stress_ratio'] = (df['monthly_repayment'] * 12) / df['annual_income']df['risk_tier'] = pd.cut(df['stress_ratio'],    bins=[0, 0.30, 0.45, 999],    labels=['Low', 'Moderate', 'Severe'])print("Risk tier distribution:")print(df['risk_tier'].value_counts())print("\nSample customers:")print(df[['name', 'email', 'risk_tier', 'stress_ratio']].head(5))df.to_csv('/content/drive/MyDrive/mortgage-agent/data/customers.csv',          index=False)print("\n[OK] Saved to Google Drive")

In [ ]:
# Cell 4 — Generate 8 weeks of weekly behavioural signalsrecords = []for week in range(8):    for _, c in df.iterrows():        stressed = c['risk_tier'] != 'Low'        # Stressed customers gradually worsen over time        m = 1 + (0.03 * week) if stressed else 1        records.append({            'customer_id': c['customer_id'],            'week': week,            'savings_balance': max(0, c['savings_balance']                - np.random.normal(500 * m, 200)),            'cc_utilisation': min(1.0,                c['credit_card_balance'] / c['credit_card_limit']                + np.random.normal(0.025 * m, 0.01)),            'days_repayment_late': int(np.random.choice(                [0, 1, 2, 3, 5, 7],                p=[.55, .15, .12, .09, .05, .04] if stressed                else [.94, .03, .01, .01, .005, .005])),            'irregular_income': bool(np.random.choice(                [True, False],                p=[.35, .65] if stressed else [.05, .95]))        })weekly = pd.DataFrame(records)weekly.to_csv('/content/drive/MyDrive/mortgage-agent/data/weekly_signals.csv',              index=False)print(f"[OK] {len(records)} weekly signal records saved")

In [ ]:
# Cell 5 — Upload to Supabase## Make sure you've already run the table-creation SQL from docs/SETUP.md# in your Supabase SQL Editor before running this cell.from supabase import create_client# Fill these in from your Supabase project settingsSUPABASE_URL = "your-supabase-url-here"SUPABASE_KEY = "your-anon-key-here"sb = create_client(SUPABASE_URL, SUPABASE_KEY)# Upload customers in batches of 100df = pd.read_csv('/content/drive/MyDrive/mortgage-agent/data/customers.csv')df['risk_tier'] = df['risk_tier'].astype('str')records = df.to_dict(orient='records')for i in range(0, len(records), 100):    sb.table('customers').upsert(records[i:i+100]).execute()print(f"[OK] {len(records)} customers uploaded")# Upload weekly signals in batches of 100weekly = pd.read_csv('/content/drive/MyDrive/mortgage-agent/data/weekly_signals.csv')sig = weekly.to_dict(orient='records')for i in range(0, len(sig), 100):    sb.table('weekly_signals').upsert(sig[i:i+100]).execute()print(f"[OK] {len(sig)} weekly signals uploaded")print("\nDatabase is seeded. Move to notebook 02_agents.ipynb next.")